# Tokens, contexto y coste: la factura de un LLM

**Facsímil 4 · La caja de herramientas** — capítulo 3 (tokens, coste, contexto y caché).

La primera sorpresa de cualquiera que pone un LLM en producción es la **factura**. No se paga por
«pregunta», se paga por **token**, y los tokens no se cuentan a ojo: un emoji puede valer varios, una
palabra rara se parte en trozos, y el español gasta más que el inglés. En este cuaderno mides, en
tokens reales, lo que de verdad procesas y pagas; calculas cuánto costaría clasificar diez mil tickets
de soporte; y entiendes por qué la **ventana de contexto** y la **caché** deciden si tu sistema es
viable o ruinoso. Saber estimar esto *antes* de gastar es puro criterio de ingeniería.

### Qué vas a aprender
- Qué es un **token** y por qué tokens ≠ palabras ≠ caracteres (con un vistazo a cómo funciona BPE).
- A **contar tokens** de cualquier texto y a estimar el **coste** de un volumen de trabajo.
- Por qué el **español** sale más caro que el inglés, en tokens.
- Qué es la **ventana de contexto** y la **caché de prefijo**, y cuánto pesan en la factura.

### Cuánto cuesta
Unos 12 minutos. CPU, sin claves.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [1]:
# En Colab:  !pip install tiktoken
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")   # el tokenizador de la familia GPT-4/3.5
def tk(texto): return enc.encode(texto)
print("Tokenizador cargado. Probemos con una frase:")
ej = "El café de la mañana me despierta."
print(f"  '{ej}'")
print(f"  -> {len(tk(ej))} tokens:", [enc.decode([t]) for t in tk(ej)])


Tokenizador cargado. Probemos con una frase:
  'El café de la mañana me despierta.'
  -> 10 tokens: ['El', ' café', ' de', ' la', ' mañana', ' me', ' des', 'pi', 'erta', '.']


## 1. Qué es un token (y por qué no es una palabra)

Un modelo no procesa letras ni palabras: procesa **tokens**, trozos de texto de tamaño variable. El
método más común para trocear, **BPE** (*byte pair encoding*), parte de los caracteres y va fusionando
los pares más frecuentes hasta formar un vocabulario de subpalabras. El resultado: las palabras
comunes son **un** token («café», «de»), las raras se parten en varios («supercalifragilístico» en
muchos), y los signos, espacios y emojis cuentan también. Por eso *tokens ≠ palabras ≠ caracteres*.
Veámoslo en una tabla.


In [2]:
ejemplos = [
    "Hola, ¿qué tal?",
    "supercalifragilisticoexpialidoso",
    "antidisestablishmentarianism",
    "def suma(a, b):\n    return a + b",
    "🙂🎉☕",
    "1234567890",
]
print(f"{'texto':<40} {'caract.':>8} {'palabras':>9} {'tokens':>7}")
for t in ejemplos:
    print(f"{t[:38]!r:<40} {len(t):>8} {len(t.split()):>9} {len(tk(t)):>7}")


texto                                     caract.  palabras  tokens
'Hola, ¿qué tal?'                              15         3       6
'supercalifragilisticoexpialidoso'             32         1      12
'antidisestablishmentarianism'                 28         1       6
'def suma(a, b):\n    return a + b'            32         7      11
'🙂🎉☕'                                           3         1       7
'1234567890'                                   10         1       4


**Lo que se ve.** Una palabra corta y común suele ser 1 token; una rara se parte en muchos; el
código y los emojis se fragmentan (un emoji puede valer varios tokens porque se codifica en varios
bytes); los números también. Esto tiene una consecuencia directa en la factura, que vemos ahora.


## 2. Español frente a inglés: la misma idea cuesta más

Los tokenizadores de los grandes modelos se entrenaron sobre todo con texto en inglés. Por eso el
inglés se «empaqueta» en menos tokens que el español: las palabras y subpalabras inglesas son más
frecuentes en el vocabulario. Comparamos la misma frase traducida.


In [3]:
pares = [
    ("Inteligencia artificial para gente curiosa", "Artificial intelligence for curious people"),
    ("El rápido zorro marrón salta sobre el perro perezoso", "The quick brown fox jumps over the lazy dog"),
]
print(f"{'español (tokens)':>20} | {'inglés (tokens)':>20} | sobrecoste ES")
for es, en in pares:
    t_es, t_en = len(tk(es)), len(tk(en))
    print(f"{es[:16]+'...':>16} {t_es:>3}  | {en[:16]+'...':>16} {t_en:>3}  | +{100*(t_es-t_en)/t_en:.0f}%")
print("\nLa misma idea en español gasta mas tokens -> mas dinero por la misma tarea.")


    español (tokens) |      inglés (tokens) | sobrecoste ES
Inteligencia art...   8  | Artificial intel...   6  | +33%
El rápido zorro ...  16  | The quick brown ...   9  | +78%

La misma idea en español gasta mas tokens -> mas dinero por la misma tarea.


## 3. La factura: diez mil tickets de soporte

Caso real: quieres clasificar y resumir los tickets de un mes. Tomamos un ticket típico, contamos sus
tokens de entrada y los de una respuesta breve, y multiplicamos. Usamos precios de **ejemplo** (los
reales varían por modelo y proveedor; cámbialos por los tuyos). La estructura del cálculo es lo que
importa.


In [4]:
ticket = ("Buenas, llevo desde ayer sin poder acceder a mi cuenta. Me dice que la "
          "contrasena es incorrecta, pero estoy seguro de que es la buena. Ya he probado "
          "a reiniciarla dos veces y no me llega el correo. Necesito entrar hoy porque "
          "tengo que descargar una factura. Gracias.")
respuesta = "Categoria: acceso. Prioridad: alta. Resumen: no recibe el correo de reinicio."
SYSTEM = "Eres un clasificador de tickets. Devuelve categoria, prioridad y resumen."

PRECIO_ENTRADA = 0.50 / 1_000_000     # $ por token de entrada (ejemplo)
PRECIO_SALIDA  = 1.50 / 1_000_000     # $ por token de salida (ejemplo)

t_in = len(tk(SYSTEM)) + len(tk(ticket))
t_out = len(tk(respuesta))
coste_uno = t_in*PRECIO_ENTRADA + t_out*PRECIO_SALIDA
print(f"Un ticket: {t_in} tokens de entrada + {t_out} de salida")
print(f"Coste de un ticket:      ${coste_uno:.6f}")
print(f"Coste de 10.000 tickets: ${coste_uno*10_000:.2f}")
print(f"Coste de 1.000.000 tickets: ${coste_uno*1_000_000:.2f}  <- a escala, cada token cuenta")


Un ticket: 88 tokens de entrada + 21 de salida
Coste de un ticket:      $0.000075
Coste de 10.000 tickets: $0.75
Coste de 1.000.000 tickets: $75.50  <- a escala, cada token cuenta


## 4. El truco de la caché: no pagar dos veces lo mismo

Fíjate en que el `SYSTEM` (las instrucciones) se repite **en cada uno** de los 10.000 tickets. Si el
proveedor cachea ese prefijo común, lo procesa una vez y no en cada llamada. Cuanto más largas sean tus
instrucciones fijas, más pesa cachearlas. Lo medimos con un *system prompt* corto y otro largo, para
ver que el ahorro crece con su longitud.


In [5]:
SYSTEM_LARGO = SYSTEM + " " + ("Clasifica segun estas reglas detalladas: " * 30)
for nombre, sysp in [("system corto", SYSTEM), ("system largo", SYSTEM_LARGO)]:
    ts = len(tk(sysp))
    sin_cache = (ts + len(tk(ticket))) * PRECIO_ENTRADA * 10_000
    con_cache = len(tk(ticket)) * PRECIO_ENTRADA * 10_000 + ts * PRECIO_ENTRADA
    print(f"{nombre} ({ts:>3} tokens): sin cache ${sin_cache:.3f} | con cache ${con_cache:.3f} "
          f"| ahorro {100*(1-con_cache/sin_cache):.0f}%")
print("\nUn system largo repetido 10.000 veces es caro; cachearlo lo convierte en pagarlo una vez.")


system corto ( 18 tokens): sin cache $0.440 | con cache $0.350 | ahorro 20%
system largo (379 tokens): sin cache $2.245 | con cache $0.350 | ahorro 84%

Un system largo repetido 10.000 veces es caro; cachearlo lo convierte en pagarlo una vez.


## 5. La ventana de contexto: cuánto cabe de una vez

Un modelo solo «ve» un número máximo de tokens a la vez (su **ventana de contexto**). Si le mandas un
PDF enorme, puede no caber, y hay que trocearlo (ahí entra el RAG, capítulo 9). Traduzcamos una ventana
a algo tangible: páginas de texto. Usamos la proporción real tokens/palabra del español que acabamos de
ver.


In [6]:
tok_por_palabra = len(tk(ticket)) / len(ticket.split())
palabras_por_pagina = 500
print(f"(En espanol, ~{tok_por_palabra:.2f} tokens por palabra.)\n")
print("ventana (tokens) | paginas aprox. | equivale a...")
for ventana, ref in [(8_000, "un articulo"), (128_000, "un libro corto"), (1_000_000, "una biblioteca pequena")]:
    pags = ventana / tok_por_palabra / palabras_por_pagina
    print(f"   {ventana:>9,}   |   {pags:>6.0f}      | {ref}")


(En espanol, ~1.46 tokens por palabra.)

ventana (tokens) | paginas aprox. | equivale a...
       8,000   |       11      | un articulo
     128,000   |      176      | un libro corto
   1,000,000   |     1371      | una biblioteca pequena


## 6. Pruébalo tú

1. **Mete tu propio texto** (un correo, un capítulo) y cuenta sus tokens *antes* de pagar por
   procesarlo. ¿Cuántas páginas de tu documento caben en una ventana de 128k?
2. **Compara idiomas:** tokeniza la misma frase en español, inglés y, si puedes, en un idioma con otro
   alfabeto. ¿Cuál gasta más? Eso es dinero.
3. **Cambia los precios** por los de tu modelo real y recalcula el millón de tickets. La diferencia
   entre un modelo y otro puede ser de 10×.
4. **Estima tu caché:** si tu *system prompt* tuviera 2.000 tokens y atendieras un millón de llamadas,
   ¿cuánto ahorrarías cacheándolo?


## 7. Errores comunes

- **Estimar tokens «a ojo» por palabras.** Te equivocas, sobre todo con código, emojis u otros idiomas.
  Cuéntalos.
- **Olvidar los tokens de salida.** Suelen costar más por token que los de entrada, y una respuesta
  larga puede dominar la factura.
- **Ignorar la caché del prefijo.** Repetir un *system prompt* largo en cada llamada es tirar dinero si
  el proveedor permite cachearlo.
- **No mirar la ventana de contexto.** Mandar más tokens de los que caben provoca errores o truncado
  silencioso; por eso existen el troceado y el RAG.


## 8. Qué te llevas

- Se paga por **tokens**, no por preguntas, y los tokens **se cuentan**, no se estiman a ojo. Mídelos
  antes de desplegar.
- El **idioma importa**: el español gasta más tokens que el inglés por la misma idea.
- La **ventana de contexto** marca cuánto cabe de una vez; la **caché de prefijo** evita pagar mil veces
  las mismas instrucciones. Estas cifras deciden si una idea es viable.

**Para seguir:** el capítulo 4 te ayuda a elegir modelo sopesando esto; los capítulos 5-6, a decidir
entre nube y local; y el RAG (capítulo 9) es, en parte, una forma de meter en la ventana solo lo justo.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*